# NeuralNav — Intent Classification: Baseline vs DistilBERT

Run this on **Kaggle** with GPU enabled: Settings -> Accelerator -> GPU T4 x2. Internet must be ON (Settings -> Internet) since this pulls the dataset from the Hugging Face Hub.

**Real data**: [Bitext Customer Service Tagged Dataset](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) — ~27 intents across 10 categories, ~26.8k labeled examples. This replaces the small hand-written `data/intents.csv` placeholder, and is large enough for the BERT fine-tune to show a real accuracy/F1 gap over the TF-IDF baseline.

Outputs at the end (download from the Kaggle Output tab into your local `models/` and `data/` folders):
- `baseline_intent.joblib`, `bert_intent/` — trained models
- `baseline_report.json`, `bert_report.json` — per-class comparison reports
- `kb.json` — auto-derived knowledge base (one representative response per intent, sourced from the dataset's own `response` column) to replace the placeholder `data/kb.json`
- `intents_real.csv` — the cleaned dataset actually used, for reproducibility

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas joblib

## Load the real dataset (Bitext, via Hugging Face Hub)

In [ ]:
import json
import pandas as pd
from datasets import load_dataset

raw = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df_full = raw["train"].to_pandas()
print(df_full.shape)
df_full.head()

In [ ]:
# Columns are: flags, instruction, category, intent, response
df = df_full.rename(columns={"instruction": "text"})[["text", "intent", "category", "response"]].dropna()
df["text"] = df["text"].str.strip()

print("Intents:", df["intent"].nunique())
print(df["intent"].value_counts())

In [ ]:
# Cap per-class examples for faster training on the free T4 quota — adjust
# MAX_PER_CLASS upward if you have GPU hours to spare and want higher accuracy.
MAX_PER_CLASS = 300
df = (
    df.groupby("intent", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), MAX_PER_CLASS), random_state=42))
    .reset_index(drop=True)
)
df.to_csv("/kaggle/working/intents_real.csv", index=False)
df.shape

## Auto-derive the knowledge base from the dataset's own responses

Picks a handful of representative (text, response) pairs per intent to seed
`kb.json` for the retrieval step in notebook 02 — replaces the hand-written
placeholder KB with answers grounded in real data.

In [ ]:
EXAMPLES_PER_INTENT_IN_KB = 3
kb = []
for intent, group in df.groupby("intent"):
    sample = group.drop_duplicates(subset="response").head(EXAMPLES_PER_INTENT_IN_KB)
    for i, row in enumerate(sample.itertuples()):
        kb.append({
            "id": f"kb_{intent}_{i}",
            "intent": intent,
            "question": row.text,
            "answer": row.response,
        })

json.dump(kb, open("/kaggle/working/kb.json", "w"), indent=2)
print(f"Built kb.json with {len(kb)} entries across {df['intent'].nunique()} intents")

## Part 1 — Classical ML baseline (TF-IDF + Logistic Regression)

In [ ]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["intent"], test_size=0.2, random_state=42, stratify=df["intent"]
)

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=2000)),
])
baseline.fit(X_train, y_train)

preds = baseline.predict(X_test)
baseline_report = classification_report(y_test, preds, output_dict=True, zero_division=0)
print(classification_report(y_test, preds, zero_division=0))

joblib.dump(baseline, "/kaggle/working/baseline_intent.joblib")
json.dump(baseline_report, open("/kaggle/working/baseline_report.json", "w"), indent=2)

## Part 2 — Fine-tune DistilBERT (uses the T4 GPU)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
import numpy as np
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

BASE_MODEL = "distilbert-base-uncased"

le = LabelEncoder()
df["label"] = le.fit_transform(df["intent"])

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["intent"])

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=48)

train_ds = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df[["text", "label"]]).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=len(le.classes_))

args = TrainingArguments(
    output_dir="/kaggle/working/_bert_ckpt",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    learning_rate=5e-5,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds)
trainer.train()

In [ ]:
preds = trainer.predict(test_ds)
pred_labels = np.argmax(preds.predictions, axis=1)
y_true = le.inverse_transform(test_df["label"].to_numpy())
y_pred = le.inverse_transform(pred_labels)

bert_report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, zero_division=0))

import os
OUT_DIR = "/kaggle/working/bert_intent"
os.makedirs(OUT_DIR, exist_ok=True)
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
json.dump(le.classes_.tolist(), open(f"{OUT_DIR}/label_classes.json", "w"))
json.dump(bert_report, open("/kaggle/working/bert_report.json", "w"), indent=2)
print("Done. Download /kaggle/working/{baseline_intent.joblib, bert_intent/, baseline_report.json, bert_report.json, kb.json, intents_real.csv}")

## Comparison table for the report

In [ ]:
comp = pd.DataFrame({
    "baseline_f1": {k: v["f1-score"] for k, v in baseline_report.items() if isinstance(v, dict)},
    "bert_f1": {k: v["f1-score"] for k, v in bert_report.items() if isinstance(v, dict)},
})
comp["delta"] = comp["bert_f1"] - comp["baseline_f1"]
comp.sort_values("delta", ascending=False)